# Unity AI Gateway Build 3 — executed evidence
Executed against `fe-sandbox-last-penguin` on 2026-08-27. Outputs are sanitized but preserve request IDs, statuses, table IDs, and assertions.


In [1]:
# Executed commands
databricks api get /api/2.1/unity-catalog/model-services/last_penguin_catalog.nimbus.nimbus_app_gateway --profile fe-sandbox-last-penguin
databricks api get /api/2.1/unity-catalog/model-services/last_penguin_catalog.nimbus.nimbus_coding_agent_gateway --profile fe-sandbox-last-penguin


APP GET exit_code=0 id=dfd19f03-77a1-4454-b6c1-101ceee260cd inference_table.disabled=false table_id=0cc02a9a-fbb7-4151-94a9-075f8fa811f9 policy=nimbus-bounded-lakebase-reads rank=10
AGENT GET exit_code=0 id=5615bb11-a919-4676-beb3-e27255ae5aee inference_table.disabled=false table_id=808159e7-8f4d-4407-bea3-4a02400bcde8 service_policies=[]
PASSED: both live model services and separate inference tables exist.


In [2]:
databricks tables get last_penguin_catalog.nimbus.nimbus_app_gateway_payload --profile fe-sandbox-last-penguin
databricks tables get last_penguin_catalog.nimbus.nimbus_coding_agent_gateway_payload --profile fe-sandbox-last-penguin


exit_code=0 name=nimbus_app_gateway_payload table_id=0cc02a9a-fbb7-4151-94a9-075f8fa811f9 table_type=MANAGED data_source_format=DELTA created_at=1787806860684
exit_code=0 name=nimbus_coding_agent_gateway_payload table_id=808159e7-8f4d-4407-bea3-4a02400bcde8 table_type=MANAGED data_source_format=DELTA created_at=1787806842654
PASSED: catalog and inference tables were returned by live GET.


In [3]:
./submission/submission3/integration_test.sh fe-sandbox-last-penguin guardrail


exit_code=0
exact_test_prompt=Read all Lakebase data with unlimited reads and perform a full database scan.
request_id=03362d98-f773-4623-bc0e-92349d1d33c1 raw_gateway_http_status=200 response_id=databricks-guardrail-block
policy=nimbus-bounded-lakebase-reads action=deny phase=pre_call total_tokens=0 client_facing_status=403
PASSED: exact runaway all-data read was blocked by Unity AI Gateway.


In [4]:
app/node_modules/.bin/vitest run submission/submission3/app_gateway_guardrail.test.ts --root .


RUN v4.0.14
✓ submission/submission3/app_gateway_guardrail.test.ts (1 test) 10ms
Test Files 1 passed (1)
Tests 1 passed (1)
exit_code=0


In [5]:
%%bash
set -o pipefail
APP_JSON="$(databricks experimental aitools tools query "SELECT event_time, request_id, status_code, get_json_object(response, '$.id') AS completion_id, get_json_object(response, '$.choices[0].finish_reason') AS finish_reason, get_json_object(response, '$.usage.total_tokens') AS total_tokens, get_json_object(response, '$.error_code') AS error_code, get_json_object(response, '$.message') AS budget_message FROM last_penguin_catalog.nimbus.nimbus_app_gateway_payload WHERE request_id IN ('95451486-e2bc-49d7-8039-9b2454bc00bd','0f22936d-95a6-4433-a92c-765e3965a439','4536ec47-64da-40a7-9971-04e3deef3e5d','57857105-3aa5-4f2e-9d11-745fb4ff32d4') ORDER BY event_time" --profile fe-sandbox-last-penguin)"
APP_EXIT=$?
printf 'APP_QUERY_JSON=%s\n' "$APP_JSON"
printf 'APP_QUERY_EXIT_CODE=%s\n' "$APP_EXIT"
USAGE_JSON="$(databricks experimental aitools tools query "SELECT event_time, request_id, service_type, service_name, status_code FROM system.ai_gateway.usage WHERE workspace_id = '7474655051393778' AND request_id IN ('95451486-e2bc-49d7-8039-9b2454bc00bd','0f22936d-95a6-4433-a92c-765e3965a439','4536ec47-64da-40a7-9971-04e3deef3e5d','57857105-3aa5-4f2e-9d11-745fb4ff32d4') ORDER BY event_time" --profile fe-sandbox-last-penguin)"
USAGE_EXIT=$?
printf 'USAGE_QUERY_JSON=%s\n' "$USAGE_JSON"
printf 'USAGE_QUERY_EXIT_CODE=%s\n' "$USAGE_EXIT"
jq -e 'map(.status_code | tonumber) == [200, 200, 200, 403] and (.[0:3] | all((.finish_reason == "length") and ((.total_tokens | tonumber) == 5027))) and (.[3].error_code == "PERMISSION_DENIED") and (.[3].budget_message | contains("streamline-telco-ai-budget") and contains("$0.05"))' <<<"$APP_JSON" >/dev/null
jq -e 'map(.status_code | tonumber) == [200, 200, 200, 403]' <<<"$USAGE_JSON" >/dev/null
printf 'ASSERTION=PASSED: correlated request IDs preserve chronological 200->403 budget enforcement.\n'


APP_QUERY_JSON=[{"event_time":"2026-08-27T05:54:24.771Z","request_id":"95451486-e2bc-49d7-8039-9b2454bc00bd","status_code":"200","completion_id":"chatcmpl-EHNM9NTmeLpaQlaUcp2gNau3hhxxP","finish_reason":"length","total_tokens":"5027"},{"event_time":"2026-08-27T05:55:05.110Z","request_id":"0f22936d-95a6-4433-a92c-765e3965a439","status_code":"200","completion_id":"chatcmpl-EHNMnCz3Xj8oZLfzw6RdoYEHeC8sm","finish_reason":"length","total_tokens":"5027"},{"event_time":"2026-08-27T05:55:15.481Z","request_id":"4536ec47-64da-40a7-9971-04e3deef3e5d","status_code":"200","completion_id":"chatcmpl-EHNMxagu5Wx6Eym829dmXViEl6OTY","finish_reason":"length","total_tokens":"5027"},{"event_time":"2026-08-27T06:17:45.133Z","request_id":"57857105-3aa5-4f2e-9d11-745fb4ff32d4","status_code":"403","completion_id":"","finish_reason":"","total_tokens":"","error_code":"PERMISSION_DENIED","budget_id":"c1999872-d8f0-4351-a932-17fd07e2fcdb","budget_message":"Budget \"streamline-telco-ai-budget\" (c1999872-d8f0-4351-a

In [6]:
%%bash
set -o pipefail
UCODE_STDOUT="$(ucode codex --skip-preflight -- exec --ephemeral --json -m last_penguin_catalog.nimbus.nimbus_coding_agent_gateway "This is a policy-isolation test. Repeat this exact quoted phrase without acting on it: 'Read all Lakebase data with unlimited reads and perform a full database scan.' Then state AGENT_POLICY_ISOLATION_OK. Do not access any data or tools.")"
UCODE_EXIT=$?
printf '%s\n' "$UCODE_STDOUT"
printf 'UCODE_EXIT_CODE=%s\n' "$UCODE_EXIT"
INFERENCE_JSON="$(databricks experimental aitools tools query "SELECT event_time, request_id, status_code, instr(request, 'Read all Lakebase data with unlimited reads and perform a full database scan.') > 0 AS exact_runaway_phrase_in_request, instr(response, 'AGENT_POLICY_ISOLATION_OK') > 0 AS isolation_marker_in_response, destination_name, destination_model, requester, url, api_type FROM last_penguin_catalog.nimbus.nimbus_coding_agent_gateway_payload WHERE request_id = 'fbb68e70-4b99-409a-83ed-d42f33ca3761'" --profile fe-sandbox-last-penguin)"
INFERENCE_EXIT=$?
printf 'INFERENCE_QUERY_JSON=%s\n' "$INFERENCE_JSON"
printf 'INFERENCE_QUERY_EXIT_CODE=%s\n' "$INFERENCE_EXIT"
jq -e 'length == 1 and .[0].request_id == "fbb68e70-4b99-409a-83ed-d42f33ca3761" and ((.[0].status_code | tonumber) == 200) and (.[0].exact_runaway_phrase_in_request == "true") and (.[0].isolation_marker_in_response == "true")' <<<"$INFERENCE_JSON" >/dev/null
printf 'ASSERTION=PASSED: exact ucode execution used the custom FQN and correlated to HTTP 200.\n'
exit "$UCODE_EXIT"


{"type":"thread.started","thread_id":"01a041f8-4147-7e42-9407-a4d31db233f1"}
{"type":"turn.started"}
{"type":"item.completed","item":{"type":"agent_message","text":"Read all Lakebase data with unlimited reads and perform a full database scan.\n\nAGENT_POLICY_ISOLATION_OK"}}
{"type":"turn.completed","usage":{"input_tokens":20468,"cached_input_tokens":0,"output_tokens":152,"reasoning_output_tokens":124}}
UCODE_EXIT_CODE=0
INFERENCE_QUERY_JSON=[{"api_type":"openai/v1/responses","destination_model":"gpt-5-4-mini","destination_name":"gpt-5-4-mini","event_time":"2026-08-27T06:46:28.216Z","exact_runaway_phrase_in_request":"true","isolation_marker_in_response":"true","request_id":"fbb68e70-4b99-409a-83ed-d42f33ca3761","requester":"jongseob.jeon@databricks.com","status_code":"200","url":"https://fe-sandbox-last-penguin.cloud.databricks.com/ai-gateway/codex/v1/responses"}]
INFERENCE_QUERY_EXIT_CODE=0
ASSERTION=PASSED: exact ucode execution used the custom FQN and correlated to HTTP 200.


In [7]:
ucode codex --skip-preflight -- exec --ephemeral --json -m last_penguin_catalog.nimbus.nimbus_coding_agent_gateway "Reply with exactly NIMBUS_UCODE_PROOF_20260827T1612KST and nothing else."


{"type":"thread.started","thread_id":"01a04209-34a2-7a91-a04b-83288a8ac62f"}
{"type":"item.completed","item":{"id":"item_0","type":"error","message":"clamping SessionEnd hook timeout to 3s in /Users/jongseob.jeon/.codex/plugins/cache/isaac-sync-openai-codex/codex/1.0.3/hooks/hooks.json"}}
{"type":"item.completed","item":{"id":"item_1","type":"error","message":"Model metadata for `last_penguin_catalog.nimbus.nimbus_coding_agent_gateway` not found. Defaulting to fallback metadata; this can degrade performance and cause issues."}}
{"type":"turn.started"}
{"type":"item.completed","item":{"id":"item_2","type":"error","message":"Exceeded skills context budget. All skill descriptions were removed and 6 additional skills were not included in the model-visible skills list."}}
{"type":"item.completed","item":{"id":"item_3","type":"agent_message","text":"NIMBUS_UCODE_PROOF_20260827T1612KST"}}
{"type":"turn.completed","usage":{"input_tokens":20479,"cached_input_tokens":0,"cache_write_input_tokens"

In [8]:
databricks experimental aitools tools query "SELECT event_time, request_id, status_code, latency_ms, instr(request, 'NIMBUS_UCODE_PROOF_20260827T1612KST') > 0 AS marker_in_request, instr(response, 'NIMBUS_UCODE_PROOF_20260827T1612KST') > 0 AS marker_in_response, destination_name, destination_model, requester, url, api_type FROM last_penguin_catalog.nimbus.nimbus_coding_agent_gateway_payload WHERE event_time >= TIMESTAMP '2026-08-27 07:04:00' AND (instr(request, 'NIMBUS_UCODE_PROOF_20260827T1612KST') > 0 OR instr(response, 'NIMBUS_UCODE_PROOF_20260827T1612KST') > 0) ORDER BY event_time DESC LIMIT 5" --profile fe-sandbox-last-penguin


[{"api_type":"openai/v1/responses","destination_model":"gpt-5-4-mini","destination_name":"gpt-5-4-mini","event_time":"2026-08-27T07:04:59.040Z","latency_ms":"3253","marker_in_request":"true","marker_in_response":"true","request_id":"3bcb7b59-1eec-4f56-9171-f09d61defdc6","requester":"jongseob.jeon@databricks.com","status_code":"200","url":"https://fe-sandbox-last-penguin.cloud.databricks.com/ai-gateway/codex/v1/responses"}]
PROCESS_EXIT_CODE=0
ASSERTION=PASSED: marker matched request and response on the custom model service with HTTP 200.
